In [ ]:
#0 Montar o Drive e apontar o dataset construído no notebook 01
# O dataset (imagens + metadata.jsonl) é produzido pelo notebook 01_dataset.ipynb.
# Aqui apenas montamos o Drive, definimos o MESMO DATA_DIR e verificamos que está pronto.
from google.colab import drive
drive.mount('/content/drive')

import os, json
DATA_DIR = '/content/drive/MyDrive/Colab Notebooks/multimodais/dados'
os.environ['DATA_DIR'] = DATA_DIR          # exportado para a célula de treino usar via "$DATA_DIR"

meta_path = os.path.join(DATA_DIR, 'metadata.jsonl')
assert os.path.exists(meta_path), f'metadata.jsonl não encontrado em {DATA_DIR}. Rode o notebook 01_dataset.ipynb primeiro.'
meta = [json.loads(l)['file_name'] for l in open(meta_path, encoding='utf-8')]
imgs = [f for f in os.listdir(DATA_DIR) if f.lower().endswith(('.jpg', '.png'))]
faltando = [m for m in meta if not os.path.exists(os.path.join(DATA_DIR, m))]
print(f'DATA_DIR = {DATA_DIR}')
print(f'Imagens: {len(imgs)} | entradas metadata: {len(meta)} | faltando: {len(faltando)}')
if faltando:
    print('  faltando:', faltando)
assert imgs and not faltando, 'Dataset incompleto. Rode/complete o notebook 01 antes de treinar.'

In [ ]:
#1 Preparação do ambiente + treino (notebook 02)

# torchao NÃO é necessário para treinar LoRA e vinha quebrado (wheel py3.10 em ambiente py3.12).
!pip uninstall -y diffusers torchao

# Dependências (aspas evitam o bug de redirecionamento do shell em ">=")
!pip -q install "transformers" "accelerate" "datasets" "peft"

# diffusers a partir do código-fonte (necessário para o script de exemplo)
![ -d /content/diffusers ] || git clone --depth 1 https://github.com/huggingface/diffusers
!pip -q install -e /content/diffusers
%cd /content/diffusers/examples/text_to_image
!pip -q install -r requirements.txt

# --- Checagens antes de gastar tempo ---
import os, torch
assert torch.cuda.is_available(), 'SEM GPU. Ambiente de execução -> Alterar tipo -> T4 GPU e rode de novo.'
assert os.environ.get('DATA_DIR'), 'Rode a célula #0 antes (ela define DATA_DIR).'
print('GPU:', torch.cuda.get_device_name(0), '| DATA_DIR:', os.environ['DATA_DIR'])

# --- Autenticação via Colab Secrets (🔑 -> criar HF_TOKEN de ESCRITA -> habilitar Notebook access) ---
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))

# --- Treinamento (usa o MESMO DATA_DIR da célula #0 via "$DATA_DIR") ---
# IMPORTANTE: --mixed_precision='fp16' precisa ir para o accelerate E para o script.
# Sem ele no script, os pesos do LoRA não são convertidos p/ fp32 e ocorre
# "Attempting to unscale FP16 gradients."
# validation_prompt em INGLÊS (mesma língua das legendas) para evitar descasamento de
# distribuição no CLIP e amostras de validação mais fiéis ao que o app vai gerar.
!accelerate launch \
  --num_processes=1 --num_machines=1 \
  --mixed_precision='fp16' --dynamo_backend='no' \
  train_text_to_image_lora.py \
  --pretrained_model_name_or_path='stable-diffusion-v1-5/stable-diffusion-v1-5' \
  --train_data_dir="$DATA_DIR" \
  --caption_column='text' \
  --mixed_precision='fp16' \
  --resolution=512 --random_flip \
  --train_batch_size=1 --gradient_accumulation_steps=4 \
  --max_train_steps=1500 --learning_rate=1e-4 --lr_scheduler='cosine' \
  --lr_warmup_steps=0 --rank=8 \
  --checkpointing_steps=500 --seed=42 \
  --output_dir='/content/atelie-lora' \
  --validation_prompt='estilo_lambelambe, a vintage black and white portrait of a woman at a sunday market' \
  --push_to_hub --hub_model_id='lamble-lambe/atelie'

In [ ]:
#2 Versionamento SEMÂNTICO da versão treinada no Hugging Face Hub (MAJOR.MINOR.PATCH)
# Fluxo: pergunta o tipo de mudança -> lê a versão que está EM PRODUÇÃO (maior tag semver no Hub)
# -> incrementa AUTOMATICAMENTE o segmento correto -> cria a nova tag apontando p/ o último commit.
import re
from huggingface_hub import HfApi
from google.colab import userdata

REPO_ID = 'lamble-lambe/atelie'
api = HfApi(token=userdata.get('HF_TOKEN'))

SEMVER = re.compile(r'^v?(\d+)\.(\d+)\.(\d+)$')

def versao_producao(repo_id):
    """Maior tag semver do repo = versão em produção (retorna (0,0,0) se não houver)."""
    try:
        tags = [t.name for t in api.list_repo_refs(repo_id).tags]
    except Exception:
        tags = []
    versoes = [tuple(map(int, m.groups())) for t in tags if (m := SEMVER.match(t))]
    return max(versoes) if versoes else (0, 0, 0)

def proxima_versao(atual, tipo):
    maj, mnr, pat = atual
    if tipo == 3:  return (maj + 1, 0, 0)       # BREAKING -> incrementa MAJOR, zera o resto
    if tipo == 2:  return (maj, mnr + 1, 0)     # FEATURE  -> incrementa MINOR, zera PATCH
    if tipo == 1:  return (maj, mnr, pat + 1)   # BUG      -> incrementa PATCH
    raise ValueError('Opção inválida (use 1, 2 ou 3).')

# 1) versão atual em produção
atual = versao_producao(REPO_ID)
print(f'Versão em produção no Hub: v{atual[0]}.{atual[1]}.{atual[2]}\n')

# 2) pergunta o tipo de mudança
print('Que tipo de mudança este treino representa?')
print('  1 = BUG       (correção)  -> PATCH   x.x.(+1)')
print('  2 = FEATURE   (melhoria)  -> MINOR   x.(+1).0')
print('  3 = BREAKING  (ruptura)   -> MAJOR   (+1).0.0')
tipo = int(input('Digite o número correspondente (1/2/3): ').strip())

# 3) calcula a nova versão e cria a tag
nova = proxima_versao(atual, tipo)
TAG = f'v{nova[0]}.{nova[1]}.{nova[2]}'
rotulo = {1: 'BUG/patch', 2: 'FEATURE/minor', 3: 'BREAKING/major'}[tipo]
desc = input(f'Descrição da {TAG} (hiperparâmetros/motivo) [enter p/ padrão]: ').strip()
DESCRICAO = desc or f'{rotulo}: rank=8, lr=1e-4, 1500 passos, 41 imagens (estilo lambe-lambe)'

print(f'\nProdução v{atual[0]}.{atual[1]}.{atual[2]}  --({rotulo})-->  {TAG}')
api.create_tag(repo_id=REPO_ID, tag=TAG, tag_message=DESCRICAO)
print(f'✅ Nova versão publicada: {TAG}')
print(f'   Hub:      https://huggingface.co/{REPO_ID}/tree/{TAG}')
print(f'   Carregar: pipe.load_lora_weights("{REPO_ID}", revision="{TAG}")')